# TASK 5 – Classification Model Evaluation and Tuning

This notebook continues the vehicle insurance fraud work from Tasks 1–4. The target is `fraud_reported_Y`, where 1 = fraud reported and 0 = not fraud.

**Task 5 checklist covered:**
- Accuracy, Precision, Recall, F1-score
- Train vs Test score for overfitting/underfitting
- 5-Fold Cross-Validation
- Comparison of all models
- GridSearchCV hyperparameter tuning
- Random Forest, AdaBoost and Gradient Boosting


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier


## 1. Load the cleaned data from Task 2

In [2]:
# Use the cleaned and encoded file produced in Task 2
df = pd.read_csv("cleaned_data.csv")
print("Shape:", df.shape)
df.head()


Shape: (12002, 45)


,claim_number,age_of_driver,safety_rating,annual_income,high_education,address_change,zip_code,claim_date,past_num_of_claims,liab_prct,...,channel_Phone,vehicle_category_Large,vehicle_category_Medium,vehicle_color_blue,vehicle_color_gray,vehicle_color_other,vehicle_color_red,vehicle_color_silver,vehicle_color_white,fraud_reported_Y
0,414724,39,73,58612.8,1,0,50048,2023-08-12,0,25,...,True,True,False,False,False,False,False,True,False,False
1,269568,33,72,35936.0,0,1,50006,NaN,0,45,...,True,False,True,False,False,False,False,False,False,True
2,974592,31,76,84940.8,1,1,15021,NaN,0,100,...,True,False,True,False,True,False,False,False,False,False
3,995328,53,93,73526.4,0,1,85027,NaN,0,100,...,True,False,True,False,False,False,True,False,False,False
4,1140480,41,87,59403.2,1,0,80046,2024-09-04,0,25,...,True,False,True,False,False,False,True,False,False,False


## 2. Prepare X and y

In [3]:
# fraud_reported_Y is the target created by get_dummies() in Task 2
X = df.drop(columns=['fraud_reported_Y', 'claim_number', 'claim_date'], errors='ignore').copy()
y = df['fraud_reported_Y'].astype(int)

# Convert boolean dummy columns to 0/1
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

# Handle any remaining missing/infinite values
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target values:")
print(y.value_counts())


X shape: (12002, 42)
y shape: (12002,)
Target values:
fraud_reported_Y
0    9051
1    2951
Name: count, dtype: int64


## 3. Train-Test Split (80% / 20%)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


Training data: (9601, 42)
Testing data: (2401, 42)


## 4. Create Classification Models

In [5]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}


## 5. Model Evaluation + Train/Test Comparison

In [6]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_score = accuracy_score(y_train, train_pred)
    test_score = accuracy_score(y_test, test_pred)

    results.append({
        'Model': name,
        'Train Accuracy': train_score,
        'Test Accuracy': test_score,
        'Accuracy': accuracy_score(y_test, test_pred),
        'Precision': precision_score(y_test, test_pred, zero_division=0),
        'Recall': recall_score(y_test, test_pred, zero_division=0),
        'F1-score': f1_score(y_test, test_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df.round(4)


,Model,Train Accuracy,Test Accuracy,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.7772,0.7759,0.7759,0.8421,0.1085,0.1922
1,Decision Tree,0.7848,0.7743,0.7743,0.8529,0.0983,0.1763
2,Random Forest,1.0000,0.7776,0.7776,0.9000,0.1068,0.1909
3,AdaBoost,0.7793,0.7776,0.7776,0.9118,0.1051,0.1884
4,Gradient Boosting,0.7852,0.7768,0.7768,0.8462,0.1119,0.1976


### Overfitting / Underfitting Check

In [7]:
for _, row in results_df.iterrows():
    difference = row['Train Accuracy'] - row['Test Accuracy']
    if difference > 0.10:
        status = 'Overfitting'
    elif row['Train Accuracy'] < 0.60 and row['Test Accuracy'] < 0.60:
        status = 'Underfitting'
    else:
        status = 'Good fit'
    print(row['Model'], '->', status, '| Train:', round(row['Train Accuracy'],4), '| Test:', round(row['Test Accuracy'],4))


Logistic Regression -> Good fit | Train: 0.7772 | Test: 0.7759
Decision Tree -> Good fit | Train: 0.7848 | Test: 0.7743
Random Forest -> Overfitting | Train: 1.0 | Test: 0.7776
AdaBoost -> Good fit | Train: 0.7793 | Test: 0.7776
Gradient Boosting -> Good fit | Train: 0.7852 | Test: 0.7768


## 6. 5-Fold Cross-Validation

In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results.append({
        'Model': name,
        'CV F1 Mean': scores.mean(),
        'CV F1 Std': scores.std(),
        'Fold 1': scores[0],
        'Fold 2': scores[1],
        'Fold 3': scores[2],
        'Fold 4': scores[3],
        'Fold 5': scores[4]
    })

cv_df = pd.DataFrame(cv_results).sort_values('CV F1 Mean', ascending=False)
cv_df.round(4)


,Model,CV F1 Mean,CV F1 Std,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5
4,Gradient Boosting,0.2103,0.0247,0.2340,0.2303,0.1856,0.2263,0.1752
2,Random Forest,0.2085,0.0220,0.2362,0.2275,0.1842,0.2127,0.1822
0,Logistic Regression,0.2077,0.0195,0.2288,0.2259,0.1757,0.2103,0.1978
1,Decision Tree,0.2034,0.0210,0.2222,0.2206,0.1780,0.2190,0.1774
3,AdaBoost,0.2026,0.0206,0.2301,0.2152,0.1752,0.2097,0.1825


**Interpretation:** Higher mean F1 is better. A smaller standard deviation means the model is more stable across folds.

## 7. Compare All Models

In [9]:
comparison_df = results_df.merge(cv_df[['Model','CV F1 Mean','CV F1 Std']], on='Model')
comparison_df = comparison_df.sort_values('F1-score', ascending=False)
comparison_df.round(4)


,Model,Train Accuracy,Test Accuracy,Accuracy,Precision,Recall,F1-score,CV F1 Mean,CV F1 Std
4,Gradient Boosting,0.7852,0.7768,0.7768,0.8462,0.1119,0.1976,0.2103,0.0247
0,Logistic Regression,0.7772,0.7759,0.7759,0.8421,0.1085,0.1922,0.2077,0.0195
2,Random Forest,1.0000,0.7776,0.7776,0.9000,0.1068,0.1909,0.2085,0.0220
3,AdaBoost,0.7793,0.7776,0.7776,0.9118,0.1051,0.1884,0.2026,0.0206
1,Decision Tree,0.7848,0.7743,0.7743,0.8529,0.0983,0.1763,0.2034,0.0210


In [10]:
# Select the model with the best cross-validation F1 score
best_model_name = cv_df.iloc[0]['Model']
print('Best model before tuning:', best_model_name)
print('CV F1 Mean:', round(cv_df.iloc[0]['CV F1 Mean'], 4))
print('CV F1 Std:', round(cv_df.iloc[0]['CV F1 Std'], 4))


Best model before tuning: Gradient Boosting
CV F1 Mean: 0.2103
CV F1 Std: 0.0247


## 8. Hyperparameter Tuning using GridSearchCV

In [11]:
if best_model_name == 'Random Forest':
    estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }
elif best_model_name == 'Gradient Boosting':
    estimator = GradientBoostingClassifier(random_state=42)
    param_grid = {
        'n_estimators': [50, 100],
        'learning_rate': [0.05, 0.1],
        'max_depth': [2, 3]
    }
elif best_model_name == 'AdaBoost':
    estimator = AdaBoostClassifier(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 150],
        'learning_rate': [0.5, 1.0]
    }
elif best_model_name == 'Decision Tree':
    estimator = DecisionTreeClassifier(random_state=42)
    param_grid = {
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_split': [2, 5, 10]
    }
else:
    estimator = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000))
    ])
    param_grid = {'model__C': [0.1, 1, 10]}

grid_search = GridSearchCV(
    estimator, param_grid, cv=cv, scoring='f1', n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Best Parameters:')
print(grid_search.best_params_)
print('Best CV F1-score:', round(grid_search.best_score_, 4))


Best Parameters:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best CV F1-score: 0.2103


## 9. Re-test the Tuned Model on Test Data

In [12]:
tuned_pred = grid_search.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_pred)
tuned_precision = precision_score(y_test, tuned_pred, zero_division=0)
tuned_recall = recall_score(y_test, tuned_pred, zero_division=0)
tuned_f1 = f1_score(y_test, tuned_pred, zero_division=0)

print('Tuned Accuracy :', round(tuned_accuracy, 4))
print('Tuned Precision:', round(tuned_precision, 4))
print('Tuned Recall   :', round(tuned_recall, 4))
print('Tuned F1-score :', round(tuned_f1, 4))


Tuned Accuracy : 0.7768
Tuned Precision: 0.8462
Tuned Recall   : 0.1119
Tuned F1-score : 0.1976


In [13]:
before_tuning_f1 = results_df.loc[results_df['Model'] == best_model_name, 'F1-score'].iloc[0]
print('F1 before tuning :', round(before_tuning_f1, 4))
print('F1 after tuning  :', round(tuned_f1, 4))

if tuned_f1 > before_tuning_f1:
    print('Result: Score improved after hyperparameter tuning.')
elif tuned_f1 == before_tuning_f1:
    print('Result: Score remained the same after hyperparameter tuning.')
else:
    print('Result: Score decreased after hyperparameter tuning.')


F1 before tuning : 0.1976
F1 after tuning  : 0.1976
Result: Score remained the same after hyperparameter tuning.


## 10. Confusion Matrix and Classification Report for Tuned Model

In [14]:
print('Confusion Matrix:')
print(confusion_matrix(y_test, tuned_pred))

print('\nClassification Report:')
print(classification_report(y_test, tuned_pred, target_names=['Not Fraud', 'Fraud'], zero_division=0))


Confusion Matrix:
[[1799   12]
 [ 524   66]]

Classification Report:
              precision    recall  f1-score   support

   Not Fraud       0.77      0.99      0.87      1811
       Fraud       0.85      0.11      0.20       590

    accuracy                           0.78      2401
   macro avg       0.81      0.55      0.53      2401
weighted avg       0.79      0.78      0.71      2401



## 11. Final Conclusion

After running the notebook, record:
1. The best model before tuning.
2. Its Accuracy, Precision, Recall and F1-score.
3. Its average 5-fold CV F1-score and standard deviation.
4. Whether it is overfitting, underfitting or a good fit.
5. The best hyperparameter combination from GridSearchCV.
6. The tuned test-set scores and whether F1 improved.
